# SCAN Py Example Usage

This notebook demonstrates the public SCAN Py API: simulation helpers, the main detector, result objects, lower-level statistics, bootstrap utilities, ensemble helpers, evaluation metrics, and plotnine visualizations.

Install the package with `pip install scan-py`. Import it as `scan`.

In [ ]:
import numpy as np

from scan import (
    BENCHMARK_CONFIG,
    adaptive_threshold,
    choose_window_sizes,
    covering_metric,
    ensemble_vote,
    f1_score_cpd,
    ipm_statistic,
    localize_cp,
    match_change_points,
    merge_change_points,
    plot_change_points,
    plot_swal_curve,
    plot_thresholds,
    plot_vote_scree,
    plot_window_votes,
    precision_recall_cpd,
    refine_cusum,
    refine_wasserstein,
    run_one_benchmark,
    safe_min_seg_len,
    scan_cpd,
    scan_single_window,
    simulate_time_series,
    tapered_block_bootstrap,
    wasserstein_statistic,
)

## 1. Simulate Data

`simulate_time_series` returns `(x, cps, means, sigmas)`:

- `x`: simulated observations
- `cps`: true change-points using Python split indexing
- `means`: segment means
- `sigmas`: segment standard deviations

In [ ]:
T = 20_000
K = 67
spacing_hint = 298
min_seg_len = safe_min_seg_len(T, K, spacing_hint)
seed = 500

window_sizes = choose_window_sizes(T, n_windows=7, seed=seed)

x, true_cps, means, sigmas = simulate_time_series(
    n=T,
    n_cps=K,
    min_seg_len=min_seg_len,
    change_type="mean",
    seed=seed,
)

x_std = (x - np.mean(x)) / np.std(x)

print(f"Simulating T={T}, K={K}, spacing_hint={spacing_hint}, min_seg_len={min_seg_len}")
print("Window sizes:", window_sizes)
print("True K:", len(true_cps))
print("First true CPs:", true_cps[:10])
print("BENCHMARK_CONFIG sample:", BENCHMARK_CONFIG[:3])

## 2. Run the Main Detector

`scan_cpd` returns a `ScanResult`. Important parameters:

- `window_sizes`: window sizes for the ensemble
- `alpha`: bootstrap tail probability
- `n_boot`: bootstrap replications
- `vote_threshold`: normalized vote cutoff
- `random_state`: reproducibility seed
- `change_type`: `"mean"`, `"var"`, or `"meanvar"`

In [ ]:
result = scan_cpd(
    x_std,
    window_sizes=window_sizes,
    n_boot=400,
    alpha=1,
    vote_threshold=0.7,
    random_state=seed,
    n_jobs=8,
    change_type="mean",
    batch_size=32,
)

print("Detected K:", len(result.change_points))
print("First detected CPs:", result.change_points[:10])
print("Elapsed seconds:", round(result.metadata["elapsed_seconds"], 4))

## 3. Inspect `ScanResult`

`ScanResult` fields:

- `change_points`: final selected change-points
- `scores`: normalized vote score per candidate
- `votes`: raw vote count per candidate
- `window_results`: per-window diagnostics
- `thresholds`: observed statistics and thresholds
- `parameters`: detector parameters
- `metadata`: runtime and backend metadata
- `segments`: merged voting segments
- `cp_dict`: property with candidates by window size

In [ ]:
print("change_points:", result.change_points[:10])
print("scores sample:", list(result.scores.items())[:5])
print("votes sample:", list(result.votes.items())[:5])
print("window_results keys:", sorted(result.window_results.keys()))
print("thresholds keys:", sorted(result.thresholds.keys()))
print("parameters:", result.parameters)
print("metadata:", result.metadata)
print("segments sample:", list(result.segments.items())[:2])
print("cp_dict sample:", {k: v[:5] for k, v in result.cp_dict.items()})

## 4. Inspect `WindowResult`

Each `WindowResult` contains diagnostics for one window size:

- `window_size`
- `change_points`
- `starts`
- `statistics`
- `lower_thresholds`
- `upper_thresholds`
- `localized_regions`

In [ ]:
first_window = window_sizes[0]
wr = result.window_results[first_window]

print("window_size:", wr.window_size)
print("window change_points:", wr.change_points[:10])
print("scan starts:", wr.starts[:10])
print("observed statistics:", wr.statistics[:5])
print("lower thresholds:", wr.lower_thresholds[:5])
print("upper thresholds:", wr.upper_thresholds[:5])
print("localized regions:", wr.localized_regions[:5])

## 5. Single-Window Scan

Use `scan_single_window` when you want to debug one window size instead of the full ensemble.

In [ ]:
single = scan_single_window(
    x_std,
    window_size=first_window,
    n_boot=100,
    alpha=1,
    random_state=seed,
    change_type="mean",
)

print("single.window_size:", single.window_size)
print("single.change_points[:10]:", single.change_points[:10])
print("len(single.statistics):", len(single.statistics))

## 6. Local Statistics and Localization

These helpers expose lower-level computations:

- `wasserstein_statistic(left, right)`
- `ipm_statistic(left, right, ipm="wasserstein")`
- `localize_cp(x, change_type="meanvar")`
- `refine_cusum(x)`
- `refine_wasserstein(x)`

In [ ]:
left = x_std[:first_window]
right = x_std[first_window : 2 * first_window]
local_block = x_std[true_cps[0] - first_window : true_cps[0] + first_window]

print("wasserstein_statistic:", wasserstein_statistic(left, right))
print("ipm_statistic:", ipm_statistic(left, right))
print("localize_cp mean:", localize_cp(local_block, change_type="mean"))
print("refine_cusum:", refine_cusum(local_block))

split, curve = refine_wasserstein(local_block)
print("refine_wasserstein split:", split)
print("first curve values:", curve[:5])

## 7. Bootstrap Utilities

- `tapered_block_bootstrap` returns bootstrap samples.
- `adaptive_threshold` returns a local bootstrap threshold for a two-window comparison.

In [ ]:
boot = tapered_block_bootstrap(
    x_std[:500],
    sample_length=100,
    block_length=None,
    n_boot=5,
    taper="tukey",
    random_state=123,
)

threshold = adaptive_threshold(
    x_std[:100],
    x_std[100:200],
    alpha=0.05,
    n_boot=100,
    random_state=123,
)

print("bootstrap shape:", boot.shape)
print("adaptive threshold:", threshold)

## 8. Ensemble Helpers

- `merge_change_points` clusters nearby candidates.
- `ensemble_vote` applies voting to a `{window_size: change_points}` mapping.

In [ ]:
clusters = merge_change_points([99, 101, 250, 260], tolerance=10)
selected, scores, votes = ensemble_vote(
    {40: [200, 400], 60: [202, 399], 80: [201]},
    vote_threshold=0.5,
    tolerance=5,
)

print("clusters:", clusters)
print("selected:", selected)
print("scores:", scores)
print("votes:", votes)

## 9. Evaluation Metrics

- `match_change_points`
- `precision_recall_cpd`
- `f1_score_cpd`
- `covering_metric`

In [ ]:
matches = match_change_points(true_cps, result.change_points, tolerance=25)
precision, recall = precision_recall_cpd(true_cps, result.change_points, tolerance=25)
f1 = f1_score_cpd(true_cps, result.change_points, tolerance=25)
covering = covering_metric(true_cps, result.change_points, n=len(x_std))

print("matches sample:", matches[:10])
print("precision:", precision)
print("recall:", recall)
print("f1:", f1)
print("covering:", covering)

## 10. Benchmark Helper

`run_one_benchmark` simulates data, runs `scan_cpd`, and returns a summary dictionary with the full `ScanResult` under `summary["result"]`.

In [ ]:
small_summary = run_one_benchmark(
    n=2000,
    k=5,
    spacing_hint=300,
    change_type="mean",
    n_boot=50,
    alpha=1,
    vote_threshold=0.5,
    n_jobs=4,
    seed=123,
)

print("summary keys:", sorted(small_summary.keys()))
print("n_detected:", small_summary["n_detected"])
print("detected sample:", small_summary["detected_cps"][:10])

## 11. Plotting

All plotting functions return plotnine `ggplot` objects. Put the plot as the last expression in a notebook cell to display it.

In [ ]:
plot_change_points(x_std, result)

In [ ]:
plot_vote_scree(result)

In [ ]:
plot_window_votes(result, max_x_labels=15, x_label_angle=45)

In [ ]:
plot_thresholds(result, window_size=first_window)

In [ ]:
if wr.localized_regions:
    start, end = wr.localized_regions[0]
    p = plot_swal_curve(x_std, start, end)
else:
    p = None
p

## 12. Save Plots

Use `.save(...)` in scripts or notebooks when you want files.

In [ ]:
from pathlib import Path

out_dir = Path("scan_plots")
out_dir.mkdir(exist_ok=True)

plot_change_points(x_std, result).save(out_dir / "change_points.png", width=11, height=4.8, dpi=150)
plot_vote_scree(result).save(out_dir / "vote_scree.png", width=8, height=4.8, dpi=150)
plot_window_votes(result).save(out_dir / "window_votes.png", width=10, height=4.8, dpi=150)
plot_thresholds(result, window_size=first_window).save(out_dir / "thresholds.png", width=10, height=4.8, dpi=150)

print("Saved plots to", out_dir.resolve())